# FreightQuote AI RAG Knowledge Base Builder
Global Logistics, Customs & Trade Compliance Knowledge Corpus.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

!pip install -q langchain langchain-community langchain-text-splitters -U langchain-core sentence-transformers faiss-cpu pymupdf beautifulsoup4 requests==2.32.4 urllib3 tqdm reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

import os
import json
import time
import requests
import urllib3
import fitz  # PyMuPDF
from bs4 import BeautifulSoup
from tqdm import tqdm
import warnings
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

warnings.filterwarnings('ignore', category=urllib3.exceptions.InsecureRequestWarning)

RAG_DIR = '/content/drive/MyDrive/FreightQuote_AI/rag_documents'
os.makedirs(RAG_DIR, exist_ok=True)
print(f'✅ Output Directory Ready: {RAG_DIR}')


In [ ]:
HTML_SOURCES = [
    "https://www.cbp.gov/trade",
    "https://www.cbp.gov/trade/basic-import-export",
    "https://www.cbp.gov/trade/priority-issues",
    "https://www.cbp.gov/trade/automated",
    "https://www.cbp.gov/trade/trade-community",
    "https://www.gov.uk/import-goods-into-uk",
    "https://www.gov.uk/export-goods",
    "https://www.gov.uk/guidance/customs-declaration-service",
    "https://www.gov.uk/government/collections/uk-trade-tariff",
    "https://taxation-customs.ec.europa.eu/customs-4/union-customs-code_en",
    "https://taxation-customs.ec.europa.eu/customs-4/customs-procedures-import_en",
    "https://taxation-customs.ec.europa.eu/customs-4/customs-procedures-export_en",
    "https://www.cbic.gov.in",
    "https://www.dgft.gov.in",
    "https://www.icegate.gov.in",
    "https://www.customs.gov.sg/businesses/importing-goods/overview",
    "https://www.customs.gov.sg/businesses/exporting-goods/overview",
    "https://www.abf.gov.au/importing-exporting-and-manufacturing/importing",
    "https://www.abf.gov.au/importing-exporting-and-manufacturing/exporting",
    "https://www.imo.org/en/OurWork/Safety/Pages/default.aspx",
    "https://www.imo.org/en/OurWork/Environment/Pages/Default.aspx",
    "https://www.iata.org/en/programs/cargo/dgr",
    "https://www.iata.org/en/programs/cargo/security",
    "http://www.wcoomd.org/en/topics/nomenclature/overview/what-is-the-harmonized-system.aspx",
    "http://www.wcoomd.org/en/topics/facilitation/overview/overview-of-trade-facilitation.aspx",
    "https://www.wto.org/english/tratop_e/tariffs_e/tariffs_e.htm",
    "https://www.wto.org/english/tratop_e/tradfa_e/tradfa_e.htm",
    "https://www.investopedia.com/terms/f/freight.asp",
    "https://www.investopedia.com/terms/p/profit-margin.asp",
    "https://www.freightos.com/freight-resources/freight-rate",
    "https://www.supplychaindigital.com",
    "https://www.logisticsmgmt.com",
    "https://iccwbo.org/business-solutions/incoterms-rules/incoterms-2020",
    "https://www.bimco.org/contracts-and-clauses/contracts",
    "https://www.fmc.gov/regulations",
    "https://marad.dot.gov",
    "https://unctad.org/topic/trade-logistics",
    "https://unctad.org/topic/trade-analysis",
    "https://www.worldbank.org/en/topic/trade"
]
HTML_SOURCES.extend([
    "https://www.bis.doc.gov/index.php/regulations/export-administration-regulations-ear",
    "https://www.bis.doc.gov/index.php/licensing",
    "https://www.trade.gov/trade-data-and-analysis",
    "https://www.trade.gov/export-solutions",
    "https://www.trade.gov/customs-broker",
    "https://www.fmc.gov/about-the-fmc",
    "https://www.fmc.gov/regulations",
    "https://www.fmc.gov/shipping-guidance",
    "https://unctad.org/topic/trade-logistics",
    "https://unctad.org/topic/trade-analysis",
    "https://www.worldbank.org/en/topic/trade",
    "https://www.worldbank.org/en/topic/transport/overview",
    "https://iccwbo.org/business-solutions/incoterms-rules/incoterms-2020",
    "https://www.intercargo.org",
    "https://www.bimco.org/contracts-and-clauses/contracts",
    "https://www.dnv.com/maritime",
    "https://www.lr.org/en/maritime",
    "https://www.marad.dot.gov",
    "https://www.gard.no/web/updates",
    "https://www.freightos.com/freight-resources",
    "https://www.supplychaindigital.com",
    "https://www.logisticsmgmt.com",
    "https://www.joc.com",
    "https://www.investopedia.com/terms/f/freight.asp",
    "https://www.investopedia.com/terms/p/profit-margin.asp",
    "https://www.classnk.or.jp/hp/en/index.html",
    "http://english.customs.gov.cn",
    "https://www.abf.gov.au/importing-exporting-and-manufacturing/tariff-classification",
    "https://www.customs.gov.sg/businesses/customs-schemes-licences-framework/overview",
    "https://taxation-customs.ec.europa.eu/customs-4/prohibitions-and-restrictions_en",
    "https://www.gov.uk/guidance/intrastat",
    "https://www.gov.uk/guidance/pay-less-import-duty-and-vat-when-re-importing-goods-to-the-uk-and-eu",
    "https://www.cbp.gov/trade/priority-issues",
    "https://www.cbp.gov/trade/automated",
    "https://www.iata.org/en/programs/cargo/live-animals",
    "https://www.iata.org/en/programs/cargo/security",
    "http://www.wcoomd.org/en/topics/enforcement-and-compliance.aspx",
    "https://www.wto.org/english/thewto_e/whatis_e/tif_e/agrm8_e.htm",
    "https://www.wto.org/english/tratop_e/tradfa_e/tradfa_e.htm"
])

PDF_SOURCES = [
    "https://unctad.org/system/files/official-document/rmt2022_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2021_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2020_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2019_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2018_en.pdf",
    "https://www.cbp.gov/sites/default/files/assets/documents/2016-Apr/icp_trade_pubs_0.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_norm/---normes/documents/publication/wcms_087817.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/36613/9781464816109.pdf"
]
PDF_SOURCES.extend([
    "https://unctad.org/system/files/official-document/rmt2022_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2021_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2020_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2019_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2018_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2017_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2016_en.pdf",
    "https://unctad.org/system/files/official-document/rmt2015_en.pdf",
    "https://unctad.org/system/files/official-document/tdr2022_en.pdf",
    "https://unctad.org/system/files/official-document/tdr2021_en.pdf",
    "https://unctad.org/system/files/official-document/tdr2020_en.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/36613/9781464816109.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/35016/9781464816123.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/33596/9781464815096.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/32436/9781464814358.pdf",
    "https://www.cbp.gov/sites/default/files/assets/documents/2016-Apr/icp_trade_pubs_0.pdf",
    "https://www.cbp.gov/sites/default/files/assets/documents/2020-Jan/Importing-into-the-United-States.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_norm/---normes/documents/publication/wcms_087817.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---travail/documents/publication/wcms_712957.pdf",
    "https://www.imo.org/en/OurWork/Safety/Documents/ISM%20Code%202014.pdf",
    "https://www.cbic.gov.in/resources//htdocs-cbec/customs/cs-act/cs-act-idx.pdf",
    "https://www.trade.gov/sites/default/files/2021-07/FTZ-Benefits-Manual.pdf",
    "https://www.bis.doc.gov/index.php/documents/regulation-docs/2339-ear-part-730/file",
    "https://www.marad.dot.gov/wp-content/uploads/pdf/MARAD_2021_USWaterborne_Full_Report.pdf",
    "https://iccwbo.org/wp-content/uploads/sites/3/2019/01/icc-incoterms-2010-publication.pdf",
    "https://www.fmc.gov/wp-content/uploads/2020/08/FMC-Shippers-Guide.pdf",
    "https://www.wcoomd.org/-/media/wco/public/global/pdf/topics/nomenclature/instruments-and-tools/hs-nomenclature-2022/hs-2022-edition-explanatory-notes.pdf",
    "https://www.wcoomd.org/-/media/wco/public/global/pdf/topics/facilitation/instruments-and-tools/declarations/kyoto-convention/kyoto_conv.pdf",
    "https://www.iata.org/contentassets/b6bc3e7b48cd4f44b7efba8ab1e4b4dc/dg-form-templates.pdf",
    "https://taxation-customs.ec.europa.eu/system/files/2021-12/ucc_work_programme_2021.pdf",
    "https://www.customs.gov.sg/files/businesses/scsb2020.pdf",
    "https://www.dgft.gov.in/CP/FTP%202023.pdf",
    "https://www.freightos.com/wp-content/uploads/2022/01/Global-Freight-Market-Report.pdf",
    "https://www.intercargo.org/wp-content/uploads/2022/10/Intercargo-Annual-Report-2022.pdf",
    "https://www.oecd-ilibrary.org/deliver/transport-outlook-2023_b6cc9ad6-en.pdf",
    "https://assets.publishing.service.gov.uk/government/uploads/system/uploads/attachment_data/file/1092198/trade-remedies-guidance.pdf",
    "https://www.abf.gov.au/importing-exporting-and-manufacturing/importing/pdf/cargo-management-modernisation-guide.pdf",
    "https://www.apec.org/docs/default-source/publications/2022/11/2022-apec-economic-policy-report/222_ec_2022-apec-economic-policy-report.pdf",
    "https://www.adb.org/sites/default/files/publication/761236/trade-finance-gaps-growth-jobs-survey-2021.pdf",
    "https://www.imf.org/~/media/Files/Publications/WEO/2022/April/English/text.ashx",
    "https://www.bimco.org/~/media/primary-toolbar/about/publications/shipping-market-review/2022/shipping-market-review-may-2022.pdf",
    "https://www.lloyds.com/news-and-insights/risk-reports/library/technology-risk/global-supply-chain.pdf",
    "https://www.wto.org/english/res_e/reser_e/ersd202111_e.pdf",
    "https://www.wto.org/english/res_e/statis_e/wts2022_e/wts2022_e.pdf",
    "https://unctad.org/system/files/official-document/ditctncd2020d3_en.pdf",
    "https://unctad.org/system/files/official-document/unctaddtlktcd20192_en.pdf",
    "https://unctad.org/system/files/official-document/unctaddtlktcd2018_en.pdf",
    "https://www.oecd.org/g20/topics/trade-and-investment/G20-trade-and-growth.pdf",
    "https://www.intracen.org/uploadedFiles/intracenorg/Content/Publications/The-State-of-Sustainable-Markets-Statistics-and-Emerging-Trends-2022.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/29971/120385.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/27509/114522.pdf",
    "https://www.oecd.org/trade/oecd-wto-aid-for-trade-at-a-glance-2019-8756cfc6-en.pdf",
    "https://www.dnv.com/binaries/content/assets/dnv/pdfs/publications/maritime-forecast-to-2050.pdf",
    "https://www.clarksons.com/media/3126/clarksons-research-shipping-review-and-outlook-spring-2022.pdf"
])


In [ ]:
# ── Additional HTML sources to significantly expand the knowledge base ──
# These are resource/publication listing pages from more countries and bodies —
# each one is likely to have several embedded PDF links, so auto-harvesting
# (harvest_pdfs_from_page) will pull in far more PDFs than the static list alone.

HTML_SOURCES.extend([
    # More national customs/trade authorities
    "https://www.cbsa-asfc.gc.ca/import/guide-eng.html",          # Canada
    "https://www.customs.go.jp/english/index.htm",                 # Japan
    "https://www.customs.go.kr/eng/main.do",                       # South Korea
    "https://www.gov.br/receitafederal/en",                        # Brazil
    "https://www.sars.gov.za/customs-and-excise/",                 # South Africa
    "https://www.dubaicustoms.gov.ae/en",                          # UAE
    "https://www.zauba.com/",                                      # India trade data
    "https://www.customs.gov.my/en",                                # Malaysia
    "https://www.customs.go.th/index.php?lang=en",                  # Thailand
    "https://www.bocgov.ph/",                                       # Philippines
    "https://www.customs.gov.vn/default.aspx?language=en-US",       # Vietnam
    "https://www.aduana.cl/aduana/site/edic/base/port/ingles.html", # Chile
    "https://www.gob.mx/tramites/ficha/importacion-definitiva/SAT2379", # Mexico

    # More port authorities & shipping bodies
    "https://www.pofr.gov.sg",
    "https://www.rotterdam.nl/en",
    "https://www.portoflosangeles.org",
    "https://www.portofrotterdam.com/en",
    "https://www.hongkongmaritimehub.com",
    "https://www.psa-international.com",
    "https://www.dpworld.com/our-business/global-network",

    # Trade & logistics publications / research bodies
    "https://www.mckinsey.com/industries/travel-logistics-and-infrastructure/our-insights",
    "https://www.weforum.org/agenda/archive/supply-chain/",
    "https://www.brookings.edu/topic/trade/",
    "https://www.piie.com/research/topics/trade-investment",
    "https://www.imf.org/en/Topics/trade",
    "https://www.oecd.org/trade/",
    "https://www.itc.int/publications",
    "https://www.wcoomd.org/en/media/newsroom.aspx",
    "https://unctad.org/topics-page/trade-and-market-access",
    "https://www.wto.org/english/res_e/publications_e/publications_e.htm",
    "https://www.adb.org/publications/series/trade-finance-gaps-growth-jobs",
    "https://www.worldbank.org/en/topic/trade/publication",

    # Freight/logistics industry news & reports
    "https://theloadstar.com",
    "https://www.joc.com/international-trade-news",
    "https://splash247.com",
    "https://www.hellenicshippingnews.com",
    "https://www.container-news.com",
    "https://safety4sea.com",
    "https://gcaptain.com",

    # Classification societies & marine insurance
    "https://www.classnk.or.jp/hp/en/publications/index.html",
    "https://www.eagle.org/en/rules-and-resources.html",
    "https://www.bureauveritas.com/en/our-business/marine-offshore",
    "https://www.rina.org/en/publications",

    # Air cargo & IATA resources
    "https://www.iata.org/en/publications/",
    "https://www.tiaca.org/news",
])
print(f"HTML_SOURCES now has {len(HTML_SOURCES)} entries (auto-discovery will pull in additional PDFs from each).")

In [ ]:
import os, json, time, requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import fitz  # PyMuPDF
from tqdm import tqdm

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

manifest_path = os.path.join(RAG_DIR, 'manifest.json')
manifest = {}
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)

def save_manifest():
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

def get_with_retry(url, max_retries=3, timeout=20):
    """GET request with exponential backoff and SSL fallback."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout, verify=True)
            resp.raise_for_status()
            return resp
        except requests.exceptions.SSLError:
            try:
                resp = requests.get(url, headers=HEADERS, timeout=timeout, verify=False)
                resp.raise_for_status()
                return resp
            except Exception as e:
                if attempt == max_retries - 1:
                    raise e
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            wait = 2 ** attempt
            time.sleep(wait)
    return None

def harvest_pdfs_from_page(url):
    """
    Visit a webpage and auto-discover all PDF links embedded inside it.
    Returns a list of absolute PDF URLs found on the page.
    """
    discovered = []
    try:
        resp = get_with_retry(url, timeout=15)
        if resp is None:
            return discovered
        soup = BeautifulSoup(resp.content, 'html.parser')
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].strip()
            # Check if link is a PDF
            if href.lower().endswith('.pdf') or '/pdf/' in href.lower():
                # Convert relative URLs to absolute
                if href.startswith('http'):
                    abs_url = href
                elif href.startswith('//'):
                    abs_url = 'https:' + href
                elif href.startswith('/'):
                    parsed = urlparse(url)
                    abs_url = f"{parsed.scheme}://{parsed.netloc}{href}"
                else:
                    abs_url = urljoin(url, href)
                abs_url = abs_url.split('#')[0]  # Remove fragments
                if abs_url not in discovered:
                    discovered.append(abs_url)
    except Exception as e:
        print(f"  ⚠️  PDF harvest failed for {url}: {e}")
    return discovered

def scrape_html(url):
    """Scrape HTML page text and save as .txt file."""
    if manifest.get(url) == 'success':
        return 'skipped'
    try:
        resp = get_with_retry(url)
        if resp is None:
            manifest[url] = 'failed: no response'
            return 'failed'
        soup = BeautifulSoup(resp.content, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        if len(text.strip()) < 100:
            manifest[url] = 'failed: too short'
            return 'failed'
        safe_name = url.replace('https://', '').replace('http://', '').replace('/', '_').replace('?', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'html_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {url}\n\n{text}')
        manifest[url] = 'success'
        save_manifest()
        return 'success'
    except Exception as e:
        manifest[url] = f'failed: {str(e)[:80]}'
        save_manifest()
        return 'failed'

def scrape_pdf(url):
    """Download a PDF and extract all text pages."""
    if manifest.get(url) == 'success':
        return 'skipped'
    try:
        resp = get_with_retry(url, timeout=45)
        if resp is None:
            manifest[url] = 'failed: no response'
            return 'failed'
        content_type = resp.headers.get('Content-Type', '')
        if 'pdf' not in content_type.lower() and not url.lower().endswith('.pdf'):
            manifest[url] = 'failed: not a pdf'
            return 'failed'
        doc = fitz.open(stream=resp.content, filetype='pdf')
        text = '\n'.join([page.get_text() for page in doc])
        doc.close()
        if len(text.strip()) < 50:
            manifest[url] = 'failed: empty pdf (scanned image)'
            return 'failed'
        safe_name = url.replace('https://', '').replace('http://', '').replace('/', '_').replace('?', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'pdf_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {url}\n\n{text}')
        manifest[url] = 'success'
        save_manifest()
        return 'success'
    except Exception as e:
        manifest[url] = f'failed: {str(e)[:80]}'
        save_manifest()
        return 'failed'

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: Scrape HTML pages AND auto-harvest embedded PDF links
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1: Scraping HTML pages + Auto-harvesting embedded PDFs')
print('=' * 60)

discovered_pdfs = set()
html_stats = {'success': 0, 'skipped': 0, 'failed': 0}

for url in tqdm(HTML_SOURCES, desc='🌐 HTML Pages'):
    result = scrape_html(url)
    html_stats[result] = html_stats.get(result, 0) + 1

    # Auto-harvest PDF links embedded in the page
    if result in ('success', 'skipped'):
        pdfs_found = harvest_pdfs_from_page(url)
        for pdf_url in pdfs_found:
            discovered_pdfs.add(pdf_url)
        if pdfs_found:
            print(f'  📎 {len(pdfs_found)} PDFs found on {url[:60]}')
    time.sleep(1)

print(f'\n✅ HTML done: {html_stats}')
print(f'📎 Auto-discovered {len(discovered_pdfs)} PDFs from HTML pages')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: Merge with static PDF_SOURCES
# ─────────────────────────────────────────────────────────────────────────────
all_pdf_urls = list(set(PDF_SOURCES) | discovered_pdfs)
print(f'\n📚 Total unique PDFs to download: {len(all_pdf_urls)}')
print(f'  → From static list:  {len(PDF_SOURCES)}')
print(f'  → Auto-discovered:   {len(discovered_pdfs)}')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: Download & extract all PDFs
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('PHASE 3: Downloading & Extracting All PDFs')
print('=' * 60)

pdf_stats = {'success': 0, 'skipped': 0, 'failed': 0}
for url in tqdm(all_pdf_urls, desc='📄 PDFs'):
    result = scrape_pdf(url)
    pdf_stats[result] = pdf_stats.get(result, 0) + 1
    if result == 'success':
        time.sleep(0.5)

print(f'\n✅ PDF done: {pdf_stats}')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
txt_files = [f for f in os.listdir(RAG_DIR) if f.endswith('.txt')]
print('\n' + '=' * 60)
print('📊 SCRAPING COMPLETE — SUMMARY')
print('=' * 60)
print(f'  HTML pages scraped:      {html_stats["success"]} success, {html_stats["skipped"]} skipped, {html_stats["failed"]} failed')
print(f'  PDFs auto-discovered:    {len(discovered_pdfs)}')
print(f'  PDFs downloaded:         {pdf_stats["success"]} success, {pdf_stats["skipped"]} skipped, {pdf_stats["failed"]} failed')
print(f'  Total .txt files in RAG: {len(txt_files)}')
print(f'  Manifest entries:        {len(manifest)}')
print('=' * 60)


In [ ]:
print(f"\nLoading {len(txt_files)} scraped text files from Drive...")
documents = []
for fname in tqdm(txt_files, desc="📂 Loading Docs"):
    filepath = os.path.join(RAG_DIR, fname)
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    if len(text.strip()) > 50:
        documents.append(Document(page_content=text, metadata={"source": fname, "type": "scraped"}))

print(f"\n✅ Loaded {len(documents)} documents into memory.")


In [ ]:
rules = [
    {"id": "CUST-101", "content": "Importing electronics into the US requires filing an Entry Summary (CBP Form 7501) and ensuring FCC compliance declaration.", "type": "rule"},
    {"id": "CUST-102", "content": "ISF 10+2 must be filed 24 hours before cargo is laden aboard a vessel destined to the United States.", "type": "rule"},
    {"id": "CUST-103", "content": "Under DDP (Delivered Duty Paid), the seller is responsible for all delivery costs including import duties and taxes.", "type": "rule"},
    {"id": "CUST-104", "content": "IMO 2020 mandates a maximum sulphur limit of 0.50% m/m (mass by mass) in fuel oil used on board ships operating outside designated emission control areas.", "type": "rule"},
    {"id": "CUST-105", "content": "Air waybills (AWB) are non-negotiable transport documents issued by the carrier.", "type": "rule"},
    {"id": "CUST-106", "content": "Customs valuation for imported goods generally uses the Transaction Value method.", "type": "rule"},
    {"id": "CUST-107", "content": "FOB (Free On Board) requires the seller to deliver the goods on board the vessel nominated by the buyer.", "type": "rule"},
    {"id": "CUST-108", "content": "EXW (Ex Works) places maximum responsibility on the buyer, who must arrange pickup and transport.", "type": "rule"},
    {"id": "CUST-109", "content": "A commercial invoice must include description of goods, value, currency, and HS code.", "type": "rule"},
    {"id": "CUST-110", "content": "Certificate of Origin is used to claim preferential tariff treatment under free trade agreements.", "type": "rule"},
    {"id": "CUST-111", "content": "Demurrage charges apply when cargo exceeds the allotted free time inside a terminal.", "type": "rule"},
    {"id": "CUST-112", "content": "Detention charges apply when cargo is picked up but the empty container is not returned within free time.", "type": "rule"},
    {"id": "CUST-113", "content": "GRI (General Rate Increase) is an adjustment of ocean freight rates applied by shipping lines.", "type": "rule"},
    {"id": "CUST-114", "content": "BAF (Bunker Adjustment Factor) is a surcharge applied to compensate for fuel price fluctuations.", "type": "rule"},
    {"id": "CUST-115", "content": "TEU (Twenty-foot Equivalent Unit) is the standard unit for measuring container capacity.", "type": "rule"},
    {"id": "CUST-116", "content": "A Letter of Credit (LC) guarantees that a buyer's payment to a seller will be received on time and for the correct amount.", "type": "rule"},
    {"id": "CUST-117", "content": "Carnet is an international customs document that permits duty-free and tax-free temporary import of goods.", "type": "rule"},
    {"id": "CUST-118", "content": "Anti-dumping duties are tariffs on imports priced below fair market value.", "type": "rule"},
    {"id": "CUST-119", "content": "Harmonized System (HS) codes are standard numeric identifiers for traded products worldwide.", "type": "rule"},
    {"id": "CUST-120", "content": "Marine insurance policies typically follow Institute Cargo Clauses A, B, or C.", "type": "rule"},
    {"id": "CUST-121", "content": "Bill of Lading mandatory fields include shipper, consignee, vessel, port, HS code, and gross weight.", "type": "rule"},
    {"id": "CUST-122", "content": "Profit margin formula for freight quotes: Base Cost + 15% Opex + 20% Target Margin.", "type": "rule"},
    {"id": "CUST-123", "content": "Human approval requirement applies for any freight quotes exceeding $50,000 USD.", "type": "rule"},
    {"id": "CUST-124", "content": "Weather-triggered re-routing: activate when wind speed > 15 m/s on the active route.", "type": "rule"},
    {"id": "CUST-125", "content": "Carbon Border Adjustment Mechanism (CBAM) applies to EU imports of cement, steel, aluminum, fertilizers, electricity starting Jan 2026.", "type": "rule"}
]

with open('kb_freight.json', 'w') as f:
    json.dump(rules, f, indent=4)

for rule in rules:
    documents.append(Document(page_content=rule["content"], metadata={"source": "kb_freight.json", "type": "rule", "id": rule["id"]}))


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} text chunks for embedding.")

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

vectorstore.save_local("freight_vectorstore")
print("FAISS vector store saved to 'freight_vectorstore'.")

In [ ]:
def query_kb(question, verbose=True):
    docs = vectorstore.similarity_search(question, k=3)
    if verbose:
        print(f"\nQ: {question}")
        for i, doc in enumerate(docs):
            source = doc.metadata.get('source', 'Unknown')
            print(f"  [{i+1}] Source: {source}\n      {doc.page_content[:200]}...")
    return docs

# ─────────────────────────────────────────────────────────────────────────
# 30+ Distinct Test Queries — spans customs, Incoterms, shipping docs,
# freight economics, IMO/safety, trade finance, and internal business rules
# ─────────────────────────────────────────────────────────────────────────
test_queries = [
    # Customs & compliance
    "What are the customs clearance steps for importing electronics into the US?",
    "What is ISF 10+2 and when must it be filed?",
    "What is required on a commercial invoice for customs purposes?",
    "What is a Certificate of Origin used for?",
    "What are Harmonized System (HS) codes?",
    "What is the Union Customs Code in the EU?",
    "What is a customs Carnet and when is it used?",
    "What are anti-dumping duties?",
    "What is the Carbon Border Adjustment Mechanism (CBAM)?",
    "What documents does the DGFT require for exports from India?",

    # Incoterms & trade terms
    "What Incoterm makes the seller responsible for all delivery costs?",
    "What does FOB mean in shipping terms?",
    "What does EXW mean and who bears the most responsibility?",
    "What is the difference between CIF and CFR?",
    "Under which Incoterm does risk transfer at the ship's rail?",

    # Shipping documents
    "What documents are mandatory on a Bill of Lading?",
    "What is an Air Waybill and is it negotiable?",
    "What is a Letter of Credit and how does it protect a seller?",
    "What is the purpose of a packing list in international shipping?",

    # Freight economics & charges
    "What is the operational cost formula for freight margin calculation?",
    "When does a freight quote require human approval?",
    "What is demurrage and when does it apply?",
    "What is the difference between demurrage and detention?",
    "What is a Bunker Adjustment Factor (BAF)?",
    "What is a General Rate Increase (GRI) in ocean freight?",
    "What does TEU stand for and why does it matter?",

    # Safety, environment & risk
    "What is the sulphur limit under IMO 2020?",
    "When should a shipment be re-routed due to weather?",
    "What are Institute Cargo Clauses in marine insurance?",
    "What safety regulations apply to dangerous goods in air cargo (IATA DGR)?",

    # Trade policy & macro context
    "What is the role of the WTO in global tariffs?",
    "What is the Harmonized System maintained by the WCO?",
    "How does UNCTAD support developing countries in trade logistics?",
    "What trends are shaping global maritime trade according to recent reports?",
]

print(f"Running {len(test_queries)} test queries against the knowledge base...\n")
print("=" * 70)

passed, failed = 0, 0
results_log = []

for q in test_queries:
    docs = query_kb(q, verbose=False)
    ok = len(docs) > 0 and all(len(d.page_content.strip()) > 20 for d in docs)
    status = "✅ PASS" if ok else "❌ FAIL"
    if ok:
        passed += 1
    else:
        failed += 1
    results_log.append((q, status, docs[0].metadata.get("source", "Unknown") if docs else "—"))
    print(f"{status} | {q}")

print("=" * 70)
print(f"\n📊 SUMMARY: {passed}/{len(test_queries)} queries passed, {failed} failed")
print(f"   Robustness rate: {passed/len(test_queries)*100:.1f}%")

# Optional: print full retrieved content for a handful of queries as a spot-check
print("\n" + "=" * 70)
print("Sample detailed results (first 3 queries):")
print("=" * 70)
for q in test_queries[:3]:
    query_kb(q, verbose=True)